In [5]:
import torch
import pandas as pd
import numpy as np
import sys

from transformers import AutoModelForSequenceClassification

sys.path.append("..")

from src.dataPreProcessing import prepareData

In [6]:
import pandas as pd
import numpy as np
import torch

processedData = pd.read_pickle("dataset/processedSplits.pkl")

xTrain = processedData["xTrain"]
yTrain = processedData["yTrain"]

xVal = processedData["xVal"]
yVal = processedData["yVal"]

xTest = processedData["xTest"]
yTest = processedData["yTest"]

labels = processedData["labels"]

print(xTrain.shape, yTrain.shape)
print(xVal.shape, yVal.shape)
print(xTest.shape, yTest.shape)

(700,) (700, 6)
(150,) (150, 6)
(150,) (150, 6)


In [7]:
tokenizer, trainLoader, valLoader, testLoader = prepareData(
    xTrain, yTrain,
    xVal, yVal,
    xTest, yTest
)

In [8]:
batch = next(iter(trainLoader))

for key, value in batch.items():
    print(key, value.shape, value.dtype)

input_ids torch.Size([8, 128]) torch.int64
attention_mask torch.Size([8, 128]) torch.int64
labels torch.Size([8, 6]) torch.float32


In [9]:
from src.model import loadModel

modelName = "distilbert-base-uncased"

model = loadModel(modelName, labels)

print("Number of labels:", model.config.num_labels)
print("Problem type:", model.config.problem_type)
print("Labels:", model.config.id2label)

c:\Users\Asus\Desktop\NLP With LLM\EndSemProject_CmntSentiAnalyser\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Asus\Desktop\NLP With LLM\EndSemProject_CmntSentiAnalyser\.venv\Lib\site-packages\huggingface_hub

Number of labels: 6
Problem type: multi_label_classification
Labels: {0: 'Hate', 1: 'Constructive_criticism', 2: 'Appreciation', 3: 'Question', 4: 'Sarcasm', 5: 'Random_unrelated'}


In [10]:
import torch
from torch.optim import AdamW

device = torch.device("cpu")

model = model.to(device)

learningRate = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=learningRate
)

criterion = torch.nn.BCEWithLogitsLoss()

print("Device:", device)
print("Optimizer:", optimizer)
print("Loss function:", criterion)

Device: cpu
Optimizer: AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 2e-05
    maximize: False
    weight_decay: 0.01
)
Loss function: BCEWithLogitsLoss()


In [11]:

from tqdm.auto import tqdm

def trainEpoch(model, trainLoader, optimizer, criterion, device):

    model.train()
    totalLoss = 0

    progressBar = tqdm(trainLoader, desc="Training")

    for batch in progressBar:

        inputIds = batch["input_ids"].to(device)
        attentionMask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=inputIds,
            attention_mask=attentionMask
        )

        # Calculate loss
        loss = criterion(outputs.logits, labels)

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        totalLoss += loss.item()

        progressBar.set_postfix(loss=loss.item())

    return totalLoss / len(trainLoader)


In [12]:

def evaluateLoss(model, valLoader, criterion, device):

    model.eval()
    totalLoss = 0

    with torch.no_grad():

        for batch in tqdm(valLoader, desc="Validation"):

            inputIds = batch["input_ids"].to(device)
            attentionMask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=inputIds,
                attention_mask=attentionMask
            )

            loss = criterion(outputs.logits, labels)

            totalLoss += loss.item()

    return totalLoss / len(valLoader)


In [13]:
import os

numEpochs = 3
bestValLoss = float("inf")

os.makedirs("../outputs/checkpoints", exist_ok=True)

for epoch in range(numEpochs):

    print(f"\nEpoch {epoch + 1}/{numEpochs}")

    trainLoss = trainEpoch(
        model, trainLoader, optimizer, criterion, device
    )

    valLoss = evaluateLoss(
        model, valLoader, criterion, device
    )

    print(f"Training Loss: {trainLoss:.4f}")
    print(f"Validation Loss: {valLoss:.4f}")

    if valLoss < bestValLoss:

        bestValLoss = valLoss

        torch.save(
            model.state_dict(),
            "../outputs/checkpoints/bestModel.pth"
        )

        print("Best model saved!")


Epoch 1/3


Validation: 100%|██████████| 19/19 [00:08<00:00,  2.19it/s]


Training Loss: 0.4747
Validation Loss: 0.4033
Best model saved!

Epoch 2/3


Validation: 100%|██████████| 19/19 [00:05<00:00,  3.33it/s]


Training Loss: 0.3713
Validation Loss: 0.3545
Best model saved!

Epoch 3/3


Validation: 100%|██████████| 19/19 [00:10<00:00,  1.82it/s]


Training Loss: 0.2889
Validation Loss: 0.3029
Best model saved!


**trainLossHistory = [0.4747, 0.3713, 0.2889]**


**valLossHistory = [0.4033, 0.3545, 0.3029]**


In [14]:
model.load_state_dict(
    torch.load(
        "../outputs/checkpoints/bestModel.pth",
        map_location=device
    )
)

model.eval()

print("Best checkpoint loaded successfully!")

Best checkpoint loaded successfully!


In [15]:
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score
)

def getPredictions(model, dataLoader, device):

    model.eval()
    allLogits = []
    allLabels = []

    with torch.no_grad():

        for batch in dataLoader:

            inputIds = batch["input_ids"].to(device)
            attentionMask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=inputIds,
                attention_mask=attentionMask
            )

            allLogits.append(outputs.logits.cpu())
            allLabels.append(batch["labels"])

    allLogits = torch.cat(allLogits)
    allLabels = torch.cat(allLabels)

    probabilities = torch.sigmoid(allLogits)

    return probabilities.numpy(), allLabels.numpy()

In [17]:
valProbabilities, valActual = getPredictions(
    model, valLoader, device
)

valPredictions = (valProbabilities >= 0.5).astype(int)

print("Probability shape:", valProbabilities.shape)
print("Prediction shape:", valPredictions.shape)
print("Actual shape:", valActual.shape)

print(classification_report(
    valActual,
    valPredictions,
    target_names=labels,
    zero_division=0
))

print("Micro F1:", f1_score(
    valActual, valPredictions, average="micro"
))

print("Macro F1:", f1_score(
    valActual, valPredictions, average="macro"
))

print("Micro Precision:", precision_score(
    valActual, valPredictions, average="micro", zero_division=0
))

print("Micro Recall:", recall_score(
    valActual, valPredictions, average="micro", zero_division=0
))

Probability shape: (150, 6)
Prediction shape: (150, 6)
Actual shape: (150, 6)
                        precision    recall  f1-score   support

                  Hate       0.00      0.00      0.00         8
Constructive_criticism       0.71      0.81      0.76        48
          Appreciation       0.82      0.69      0.75        45
              Question       1.00      0.37      0.54        19
               Sarcasm       0.00      0.00      0.00        23
      Random_unrelated       0.00      0.00      0.00        13

             micro avg       0.77      0.49      0.60       156
             macro avg       0.42      0.31      0.34       156
          weighted avg       0.58      0.49      0.51       156
           samples avg       0.51      0.49      0.50       156

Micro F1: 0.6015625
Macro F1: 0.34045517394447095
Micro Precision: 0.77
Micro Recall: 0.4935897435897436


In [18]:
for i, label in enumerate(labels):

    print(f"\n{label}")

    print("Actual positives:", int(valActual[:, i].sum()))

    print("Predicted positives:",
          int(valPredictions[:, i].sum()))

    print("Mean probability:",
          round(valProbabilities[:, i].mean(), 4))

    print("Maximum probability:",
          round(valProbabilities[:, i].max(), 4))

    print("90th percentile:",
          round(np.percentile(valProbabilities[:, i], 90), 4))


Hate
Actual positives: 8
Predicted positives: 0
Mean probability: 0.0942
Maximum probability: 0.2841
90th percentile: 0.188

Constructive_criticism
Actual positives: 48
Predicted positives: 55
Mean probability: 0.3913
Maximum probability: 0.9046
90th percentile: 0.8623

Appreciation
Actual positives: 45
Predicted positives: 38
Mean probability: 0.2979
Maximum probability: 0.8989
90th percentile: 0.8188

Question
Actual positives: 19
Predicted positives: 7
Mean probability: 0.0927
Maximum probability: 0.799
90th percentile: 0.1665

Sarcasm
Actual positives: 23
Predicted positives: 0
Mean probability: 0.1705
Maximum probability: 0.4616
90th percentile: 0.295

Random_unrelated
Actual positives: 13
Predicted positives: 0
Mean probability: 0.0974
Maximum probability: 0.3442
90th percentile: 0.1606


In [21]:

from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.05, 0.51, 0.01)

bestThresholds = []
bestF1Scores = []

for i, label in enumerate(labels):

    bestF1 = -1
    bestThreshold = 0.5

    for threshold in thresholds:

        predictions = (
            valProbabilities[:, i] >= threshold
        ).astype(int)

        score = f1_score(
            valActual[:, i],
            predictions,
            zero_division=0
        )

        if score > bestF1:
            bestF1 = score
            bestThreshold = threshold

    bestThresholds.append(bestThreshold)
    bestF1Scores.append(bestF1)

    print(
        f"{label}: Threshold={bestThreshold:.2f}, "
        f"F1={bestF1:.3f}"
    )

valPredictionsTuned = (
    valProbabilities >= np.array(bestThresholds)
).astype(int)


print(classification_report(
    valActual,
    valPredictionsTuned,
    target_names=labels,
    zero_division=0
))

print("Tuned Macro F1:",
      f1_score(valActual, valPredictionsTuned, average="macro"))

print("Tuned Micro F1:",
      f1_score(valActual, valPredictionsTuned, average="micro"))


Hate: Threshold=0.21, F1=0.471
Constructive_criticism: Threshold=0.50, F1=0.757
Appreciation: Threshold=0.44, F1=0.776
Question: Threshold=0.13, F1=0.778
Sarcasm: Threshold=0.21, F1=0.459
Random_unrelated: Threshold=0.17, F1=0.500
                        precision    recall  f1-score   support

                  Hate       0.44      0.50      0.47         8
Constructive_criticism       0.71      0.81      0.76        48
          Appreciation       0.82      0.73      0.78        45
              Question       0.82      0.74      0.78        19
               Sarcasm       0.37      0.61      0.46        23
      Random_unrelated       0.55      0.46      0.50        13

             micro avg       0.65      0.71      0.67       156
             macro avg       0.62      0.64      0.62       156
          weighted avg       0.68      0.71      0.69       156
           samples avg       0.63      0.70      0.65       156

Tuned Macro F1: 0.6235224246913118
Tuned Micro F1: 0.674846625

In [22]:
testProbabilities, testActual = getPredictions(
    model, testLoader, device
)

testPredictions = (
    testProbabilities >= np.array(bestThresholds)
).astype(int)

print(classification_report(
    testActual,
    testPredictions,
    target_names=labels,
    zero_division=0
))

print("Test Macro F1:",
      f1_score(testActual, testPredictions, average="macro"))

print("Test Micro F1:",
      f1_score(testActual, testPredictions, average="micro"))

                        precision    recall  f1-score   support

                  Hate       0.25      0.50      0.33         8
Constructive_criticism       0.71      0.86      0.78        49
          Appreciation       0.80      0.72      0.76        46
              Question       0.74      0.74      0.74        19
               Sarcasm       0.40      0.67      0.50        24
      Random_unrelated       0.56      0.38      0.45        13

             micro avg       0.62      0.72      0.66       159
             macro avg       0.58      0.64      0.59       159
          weighted avg       0.66      0.72      0.68       159
           samples avg       0.63      0.69      0.64       159

Test Macro F1: 0.5935198934291493
Test Micro F1: 0.6647230320699709


In [24]:
import os
import json

finalModelDir = "../outputs/finalModel"
os.makedirs(finalModelDir, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(finalModelDir)
tokenizer.save_pretrained(finalModelDir)

# Save label-specific thresholds
thresholdData = {
    label: float(threshold)
    for label, threshold in zip(labels, bestThresholds)
}

with open(os.path.join(finalModelDir, "thresholds.json"), "w") as file:
    json.dump(thresholdData, file, indent=4)

print("Final model saved successfully!")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.63it/s]

Final model saved successfully!


In [27]:
import os

print("Current working directory:", os.getcwd())
print("Final model absolute path:", os.path.abspath(finalModelDir))
print("Contents of current directory:", os.listdir())

Current working directory: c:\Users\Asus\Desktop\NLP With LLM\EndSemProject_CmntSentiAnalyser
Final model absolute path: c:\Users\Asus\Desktop\NLP With LLM\EndSemProject_CmntSentiAnalyser\outputs\finalModel
Contents of current directory: ['.venv', 'dataset', 'notebooks', 'outputs', 'src']


In [26]:
finalModelDir = "outputs/finalModel"
os.makedirs(finalModelDir, exist_ok=True)

model.save_pretrained(finalModelDir)
tokenizer.save_pretrained(finalModelDir)

with open(os.path.join(finalModelDir, "thresholds.json"), "w") as file:
    json.dump(thresholdData, file, indent=4)

print("Final model saved at:", os.path.abspath(finalModelDir))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.23it/s]

Final model saved at: c:\Users\Asus\Desktop\NLP With LLM\EndSemProject_CmntSentiAnalyser\outputs\finalModel
